# Winter wheat yield — TorchCrop against CyBench

**What this notebook does.** It aggregates the 10 km TorchCrop winter-wheat
yield simulation to national averages, compares them with the CyBench
sub-national yield statistics aggregated the same way, and reports RMSE, MAE,
Bias, MAPE, R² and Pearson r per country and pooled.

**Scope.** EU-27 and Schengen, excluding Ukraine and Russia. CyBench publishes
wheat for 23 of those 31 countries; the rest are listed in §1 and dropped.

**Three conventions worth knowing before reading any number:**

1. **Error is `simulated − observed`.** A negative bias means the model is low.
2. **No moisture conversion is applied.** TorchCrop's LINTUL-5 reports grain
   **dry matter**; CyBench reports the national statistics at market moisture
   (~13.5 % for wheat). Both sides are compared as published, so a systematic
   offset of roughly that size is expected on top of any model error. To put
   both on dry matter instead, call `aggregate.to_dry_matter` on the
   observations — everything downstream is unchanged.
3. **The simulated national mean is an unweighted mean of cropland cells.** The
   SIMPLACE export carries no per-cell wheat area, so no weight can be applied
   without inventing one. The observed national mean *is* area-weighted, by
   `harvest_area`, wherever enough regions report it.

All reusable code is in [`utils/`](utils/); this notebook is the workflow only.

## 0. Setup

In [ ]:
import logging
import sys
from pathlib import Path

# cropmodelling4eu is installed (pip install -e .), so the evaluation
# library is imported like any other package rather than off sys.path.

import numpy as np
import pandas as pd

from cropmodelling4eu.evaluation import aggregate, config, cybench, doy, metrics, plots, regions, torchcrop
from cropmodelling4eu.evaluation.style import use_style

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s",
                    force=True)
logging.getLogger("matplotlib").setLevel(logging.WARNING)

PALETTE = use_style("light")
config.ensure_output_dirs()

pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 140)

print(f"TorchCrop run : {config.TORCHCROP_RUN_DIR}")
print(f"CyBench root  : {config.CYBENCH_ROOT}")
print(f"Outputs       : {config.OUTPUT_DIR}")

## 1. Scope — which countries are evaluable

A country needs a CyBench yield file *and* CyBench polygons: the file supplies
the reference, the polygons decide which 10 km cells belong to it.

In [ ]:
COUNTRIES = config.available_countries()

scope = pd.DataFrame({
    "code": list(config.TARGET_COUNTRIES),
    "name": [config.COUNTRY_NAMES.get(c, c) for c in config.TARGET_COUNTRIES],
    "eu27": [c in config.EU27 for c in config.TARGET_COUNTRIES],
    "schengen_non_eu": [c in config.SCHENGEN_NON_EU for c in config.TARGET_COUNTRIES],
    "in_cybench": [c in COUNTRIES for c in config.TARGET_COUNTRIES],
}).set_index("code")

print(f"{len(COUNTRIES)} of {len(config.TARGET_COUNTRIES)} target countries are evaluable")
print("missing:", ", ".join(scope.index[~scope["in_cybench"]]))
scope

## 2. Load the TorchCrop simulation

One row per (cell, season). `year` is the **harvest** year — winter wheat is
sown in the preceding autumn.

In [ ]:
sim = torchcrop.load_simulation(columns=[
    "SimplaceID", "year", "lon", "lat", "yield_t_ha", "biomass_g_m2",
    "max_lai", "days_to_maturity", "tranrf_mean", "nni_mean",
    "heat_stress_factor", "n_applied_g_m2", "irri",
])

print(f"{len(sim):,} cell-seasons, {sim['SimplaceID'].nunique():,} cells, "
      f"{sim['year'].min()}-{sim['year'].max()}")
sim[["yield_t_ha", "biomass_g_m2", "max_lai", "days_to_maturity"]].describe().T

## 3. Assign each 10 km cell to a country

The national footprints are the CyBench administrative polygons dissolved per
country, so both sides of the comparison cover the same ground. Cells outside
that footprint — the UK, Norway, Switzerland, the western Balkans, North
Africa — are dropped.

In [ ]:
cells = torchcrop.simulation_cells(sim)
cells = regions.assign_cells_to_countries(
    cells, COUNTRIES, cache=regions.default_cache_path(len(COUNTRIES))
)

sim = sim.merge(cells[["SimplaceID", "country", "snapped"]], on="SimplaceID", how="left")
scoped = sim[sim["country"].notna()].copy()

print(f"{cells['country'].notna().sum():,} of {len(cells):,} cells inside the "
      f"CyBench footprint ({cells['snapped'].sum():,} matched by the {config.SNAP_KM} km snap)")
print(f"{len(scoped):,} of {len(sim):,} simulated cell-seasons kept")

cells_per_country = (
    cells.dropna(subset=["country"]).groupby("country").size().rename("cells").to_frame()
)
cells_per_country["share_%"] = (
    100 * cells_per_country["cells"] / cells_per_country["cells"].sum()
).round(1)
cells_per_country.T

## 4. Aggregate the simulation to national means

In [ ]:
sim_country = aggregate.aggregate_simulated(scoped, {"yield_t_ha": False})
sim_country.head()

## 5. Load and aggregate the CyBench observations

Weighted by `harvest_area` wherever at least half a country's regions report
one, which makes the national value `Σ production / Σ area` rather than the
mean of regional yields. Where it does not, the aggregation falls back to the
unweighted mean and records that in `obs_method` — Germany reports no area for
three quarters of its rows.

In [ ]:
obs = cybench.load_yield(COUNTRIES)
obs_country = aggregate.aggregate_observed_yield(obs)

print(obs_country["obs_method"].value_counts().to_string())
obs_country.head()

## 6. Pair the two sides

An inner join on `(country, year)`. What it drops is logged: simulated years
CyBench does not cover (2021–2024 for most countries) and observed years before
the run starts (pre-2000).

In [ ]:
paired = aggregate.pair_observations(sim_country, obs_country, ["country", "year"])
paired["residual"] = paired["yield_t_ha"] - paired["obs_yield"]

coverage = (
    paired.groupby("country")
    .agg(n_years=("year", "size"), first=("year", "min"), last=("year", "max"),
         cells=("n_cells", "median"), regions=("n_regions", "median"))
    .sort_values("n_years")
)
print(f"{len(paired)} paired country-years across {paired['country'].nunique()} countries")
coverage.T

## 7. Metrics

`R²` is the coefficient of determination `1 − SS_res/SS_tot`, **not** the square
of Pearson r — it goes negative when the simulation predicts worse than the
observed mean, which is the informative case for a process model carrying a
systematic offset. Pearson r is reported separately, so a country whose
interannual pattern is right but whose level is wrong is still visible.

In [ ]:
pooled = metrics.yield_metrics(paired["obs_yield"], paired["yield_t_ha"])
print("Pooled over every country-year:")
for key in metrics.YIELD_METRIC_ORDER:
    print(f"  {metrics.METRIC_LABELS[key]:>16s}  {pooled[key]:>8.2f}")

In [ ]:
by_country = metrics.metrics_by_group(paired, "obs_yield", "yield_t_ha")
ranked = metrics.rank_countries(by_country, by="rmse")

ranked.drop(columns=["sparse", "pearson_p"]).to_csv(
    config.TABLE_DIR / "yield_metrics_by_country.csv", float_format="%.3f"
)
plots.metric_table(
    ranked, ("rank", *metrics.YIELD_METRIC_ORDER),
    caption="Country yield skill, ranked by RMSE (best first). "
            "Bias and R² are in t/ha and dimensionless; countries with fewer "
            "than three paired years are listed last, unranked.",
)

### Ranked by correlation instead

RMSE ranks by usability. Pearson r ranks by whether the model tracks the *shape*
of the interannual variation, which is a different question and gives a
different order.

In [ ]:
plots.metric_table(
    metrics.rank_countries(by_country, by="pearson_r", ascending=False),
    ("rank", "n", "pearson_r", "pearson_p", "bias", "rmse"),
    gradient_on=("pearson_r",),
    caption="The same countries ranked by interannual correlation.",
)

## 8. Figures

### 8.1 Observed against simulated

Every country-year in one cloud. With 23 countries on screen colour cannot
carry identity — the categorical palette separates at most eight hues — so the
points share one hue and the countries are separated by panel in §8.2 instead.
The three largest under-predictions are highlighted.

In [ ]:
worst = ranked.drop(index="ALL").nsmallest(3, "bias").index.tolist()

fig = plots.scatter_one_to_one(
    paired, "obs_yield", "yield_t_ha", stats=pooled,
    title="National winter wheat yield, 2000-2020",
    xlabel="CyBench observed yield (t ha$^{-1}$)",
    ylabel="TorchCrop simulated yield (t ha$^{-1}$)",
    highlight=worst,
)
plots.save(fig, "yield_01_scatter_one_to_one")
fig

### 8.2 One panel per country

Shared axes across every panel, so a country's distance from the diagonal is
comparable at a glance. Panels are ordered by RMSE, best first.

In [ ]:
fig = plots.scatter_small_multiples(
    paired, "obs_yield", "yield_t_ha", metrics=by_country,
    order=[c for c in ranked.index if c != "ALL"],
    title="Observed against simulated yield, by country (ordered by RMSE)",
    xlabel="CyBench observed yield (t ha$^{-1}$)",
    ylabel="TorchCrop simulated yield (t ha$^{-1}$)",
)
plots.save(fig, "yield_02_scatter_small_multiples")
fig

### 8.3 Country-wise bias

The fill carries the **sign** only — the bar's length already encodes
magnitude. The whisker is ± RMSE, which is always at least |bias|; where it is
much larger, the country's error is scatter rather than offset.

In [ ]:
fig = plots.bias_bars(
    ranked, value_col="bias", error_col="rmse",
    title="Mean yield bias by country (simulated - observed)",
    xlabel="Bias (t ha$^{-1}$)",
)
plots.save(fig, "yield_03_bias_by_country")
fig

### 8.4 Residuals

Three views, because they fail differently: against the observation (a slope
error shows as a trend), against time (drift or a bad year), and as a
distribution (shift versus spread).

In [ ]:
fig = plots.residual_panels(
    paired, "residual", "obs_yield",
    title="Yield residuals",
    ylabel="Residual (t ha$^{-1}$)",
    xlabel_obs="CyBench observed yield (t ha$^{-1}$)",
)
plots.save(fig, "yield_04_residuals")
fig

### 8.5 Time series by country

In [ ]:
fig = plots.timeseries_small_multiples(
    paired, "obs_yield", "yield_t_ha",
    order=[c for c in ranked.index if c != "ALL"],
    title="National yield through time",
    ylabel="Yield (t ha$^{-1}$)",
)
plots.save(fig, "yield_05_timeseries")
fig

### 8.6 Maps

The native 10 km field first — the country means above are averages over this —
then the country bias.

In [ ]:
polygons = regions.load_country_polygons(COUNTRIES)

cell_mean = (
    scoped.groupby(["SimplaceID", "lon", "lat"], as_index=False)["yield_t_ha"].mean()
)

fig = plots.cell_map(
    cell_mean, "yield_t_ha",
    title=f"Simulated mean winter wheat yield, {paired['year'].min()}-{paired['year'].max()}",
    cbar_label="Yield (t ha$^{-1}$)", overlay=polygons,
)
plots.save(fig, "yield_06_map_simulated_cells")
fig

In [ ]:
fig = plots.country_choropleth(
    polygons, ranked["bias"],
    title="Mean yield bias (simulated - observed)",
    cbar_label="Bias (t ha$^{-1}$)", diverging=True,
)
plots.save(fig, "yield_07_map_bias")
fig

## 9. Diagnostics — where the failure is, and what it looks like

The bias is not a uniform offset: it is near zero across the Baltic and central
Europe and catastrophic around the North Sea. That pattern is not something the
metrics explain on their own, so this section pulls the run's own state
variables alongside the bias to say what the model is doing in the countries it
gets wrong.

In [ ]:
state = (
    scoped.groupby("country")
    .agg(sim_yield=("yield_t_ha", "mean"),
         max_lai=("max_lai", "mean"),
         biomass=("biomass_g_m2", "mean"),
         days_to_maturity=("days_to_maturity", "mean"),
         water_stress=("tranrf_mean", "mean"),
         n_index=("nni_mean", "mean"),
         heat=("heat_stress_factor", "mean"),
         failed_share=("yield_t_ha", lambda s: float((s < 0.5).mean())))
)
state["obs_yield"] = ranked["obs_mean"]
state["bias"] = ranked["bias"]

print("Correlation of country bias with the run's own state variables:")
print(state.corr(numeric_only=True)["bias"].drop("bias").round(2).sort_values().to_string())

state.sort_values("bias").round(3)

`failed_share` is the fraction of cell-seasons yielding under 0.5 t ha⁻¹ — an
effective crop failure. Read it next to `max_lai`: where the simulated canopy
never closes, there is no yield to speak of, and the country's bias is that
failure rather than a calibration offset.

In [ ]:
fig = plots.cell_map(
    scoped.groupby(["SimplaceID", "lon", "lat"], as_index=False)["max_lai"].mean(),
    "max_lai",
    title="Simulated maximum leaf area index (mean over seasons)",
    cbar_label="Max LAI (m$^2$ m$^{-2}$)", overlay=polygons,
)
plots.save(fig, "yield_08_map_max_lai")
fig

## 10. Summary

In [ ]:
summary = pd.Series({
    "countries": paired["country"].nunique(),
    "country-years": len(paired),
    "years": f"{paired['year'].min()}-{paired['year'].max()}",
    "10 km cells": int(cells["country"].notna().sum()),
    "observed mean (t/ha)": round(pooled["obs_mean"], 2),
    "simulated mean (t/ha)": round(pooled["sim_mean"], 2),
    "bias (t/ha)": round(pooled["bias"], 2),
    "RMSE (t/ha)": round(pooled["rmse"], 2),
    "MAE (t/ha)": round(pooled["mae"], 2),
    "MAPE (%)": round(pooled["mape"], 1),
    "R2": round(pooled["r2"], 2),
    "Pearson r": round(pooled["pearson_r"], 2),
}, name="value")

paired.to_csv(config.TABLE_DIR / "yield_paired_country_year.csv",
              index=False, float_format="%.4f")
state.to_csv(config.TABLE_DIR / "yield_diagnostics_by_country.csv",
             float_format="%.4f")
print(f"tables written to {config.TABLE_DIR}")
print(f"figures written to {config.FIGURE_DIR}")
summary.to_frame()

### What the numbers mean, and what they do not

* **The pooled skill is dominated by a level error, not by noise.** The bias is
  a large share of the RMSE, and the residual trends with the observation — the
  model reproduces less of the high-yielding end than of the low.
* **Part of that offset is a units difference, not model error.** No moisture
  conversion was applied (see the header); the simulated dry matter is expected
  to sit ~13.5 % below a market-moisture statistic before any model error is
  counted. That accounts for a fraction of the offset, not for the countries
  simulating near-total crop failure.
* **The maritime north-west is a distinct failure, not a worse calibration.**
  §9 shows the simulated canopy never developing there, so those countries'
  metrics measure a model failure, not a yield gap. Excluding them changes the
  pooled numbers substantially — quote the per-country table, not the pooled
  row, for anything downstream.
* **The observed side has its own limits.** CyBench's national averages come
  from sub-national statistics with uneven area reporting; `obs_method` records
  where the weighting fell back.